# Lectures 12–13: Conditioning & Floating Point Arithmetic
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Lecture 12: Conditioning

### Polynomial Root Sensitivity (Wilkinson Polynomial)

The polynomial with roots at $1, 2, \ldots, 20$ has massively ill-conditioned roots.

In [ ]:
# Build the Wilkinson polynomial: roots at 1, 2, ..., 20
roots_true = np.arange(1, 21, dtype=float)
coeffs = np.poly(roots_true)  # polynomial coefficients (high degree first)

plt.figure(figsize=(8, 6))
plt.plot(roots_true, np.zeros_like(roots_true), 'ko', markersize=6, label='True roots')

for _ in range(500):
    # Relative perturbation of 1e-10 in coefficients
    perturbed = coeffs * (1 + 1e-10 * np.random.randn(len(coeffs)))
    r = np.roots(perturbed)
    plt.plot(r.real, r.imag, '.', color='steelblue', markersize=2, alpha=0.3)

plt.xlabel('Re')
plt.ylabel('Im')
plt.title('Wilkinson polynomial: roots under $10^{-10}$ relative perturbation')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Even without perturbation, computed roots are inaccurate
r_computed = np.sort(np.roots(coeffs).real)
print("Computed roots of Wilkinson polynomial (no perturbation):")
for i, (true, comp) in enumerate(zip(roots_true, r_computed)):
    err = abs(comp - true)
    if err > 1e-10:
        print(f"  root {i+1:2d}: true = {true:5.1f}, computed = {comp:.6f}, error = {err:.2e}  ← inaccurate")
    else:
        print(f"  root {i+1:2d}: true = {true:5.1f}, computed = {comp:.10f}")

### Scalar Conditioning Examples

In [ ]:
# f(x) = x^2 at x = 3: kappa = 2
x = 3.0
delta = 0.01  # 1% perturbation
f_exact = x**2
f_pert = (x * (1 + delta))**2
rel_change_out = abs(f_pert - f_exact) / abs(f_exact)
print(f"f(x) = x^2 at x = 3:")
print(f"  kappa = 2")
print(f"  1% input perturbation -> {100*rel_change_out:.2f}% output change\n")

# f(x) = x - 1 at x = 1.00001: kappa ~ 100000
x = 1.00001
kappa = abs(x) / abs(x - 1)
print(f"f(x) = x - 1 at x = 1.00001:")
print(f"  kappa = {kappa:.0f}")
print(f"  A relative perturbation of 1e-10 causes relative output change ~ {kappa * 1e-10:.2e}\n")

# f(x) = sqrt(x) at x = 4: kappa = 1/2
x = 4.0
f_exact = np.sqrt(x)
f_pert = np.sqrt(x * 1.01)
rel_change_out = abs(f_pert - f_exact) / abs(f_exact)
print(f"f(x) = sqrt(x) at x = 4:")
print(f"  kappa = 1/2")
print(f"  1% input perturbation -> {100*rel_change_out:.2f}% output change")

### Matrix Condition Number: Perturbation Experiment

In [ ]:
from scipy.linalg import hilbert

A = hilbert(5)
kappa = np.linalg.cond(A)
print(f"Hilbert(5): kappa = {kappa:.4e}\n")

x_true = np.array([1.3, 2.3, 3.3, 4.3, 5.3])
b = A @ x_true

# Perturb b
print("Perturbing b by relative 1e-10:")
for _ in range(8):
    bb = b * (1 + 1e-10 * (2 * np.random.rand(len(b)) - 1))
    x_comp = np.linalg.solve(A, bb)
    rel_err = np.linalg.norm(x_comp - x_true) / np.linalg.norm(x_true)
    print(f"  relative error = {rel_err:.2e}")
print(f"  bound = {1e-10 * kappa:.4e}")

In [ ]:
# Perturb A
print("Perturbing A by relative 1e-10:")
for _ in range(8):
    AA = A * (1 + 1e-10 * (2 * np.random.rand(*A.shape) - 1))
    x_comp = np.linalg.solve(AA, b)
    rel_err = np.linalg.norm(x_comp - x_true) / np.linalg.norm(x_true)
    print(f"  relative error = {rel_err:.2e}")
print(f"  bound = {1e-10 * kappa:.4e}")

### Hilbert Matrix: Condition Number Growth

In [ ]:
sizes = range(2, 21)
conds = [np.linalg.cond(hilbert(n)) for n in sizes]

plt.figure(figsize=(8, 4.5))
plt.semilogy(list(sizes), conds, 'ro-', markersize=5)
plt.axhline(1 / np.finfo(float).eps, color='gray', linestyle='--', alpha=0.5,
            label=r'$1/\epsilon_{\mathrm{machine}}$ (hopeless beyond here)')
plt.xlabel('$n$')
plt.ylabel(r'$\kappa(H_n)$')
plt.title('Hilbert matrix condition number')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

for n in [5, 10, 15, 20]:
    k = np.linalg.cond(hilbert(n))
    digits = max(0, 16 - np.log10(k))
    print(f"n={n:2d}: kappa = {k:.2e}, ~{digits:.0f} correct digits")

### Geometry of Ill-Conditioning

In [ ]:
# Well-conditioned vs ill-conditioned: unit circle -> ellipse
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t = np.linspace(0, 2*np.pi, 300)
circle = np.array([np.cos(t), np.sin(t)])

for ax, A_mat, title in zip(axes,
    [np.array([[1, 0], [0, 0.8]]),
     np.array([[1, 1], [1, 1.00001]])],
    [r'Well-conditioned ($\kappa \approx 1.25$)',
     r'Ill-conditioned ($\kappa \approx 4 \times 10^5$)']):

    ellipse = A_mat @ circle
    kap = np.linalg.cond(A_mat)

    ax.plot(circle[0], circle[1], 'b-', alpha=0.3, label='Unit circle')
    ax.plot(ellipse[0], ellipse[1], 'r-', linewidth=2, label='$A$ · (unit circle)')
    ax.set_aspect('equal')
    ax.set_title(f'{title}\n$\\kappa = {kap:.2e}$', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle('Conditioning = shape of the ellipsoid', fontsize=13)
plt.tight_layout()
plt.show()

---
## Lecture 13: Floating Point Arithmetic

### Machine Epsilon and IEEE Double Precision

In [ ]:
import struct

eps = np.finfo(np.float64).eps
print(f"Machine epsilon (float64): {eps}")
print(f"  = 2^(-52) = {2**-52}")
print(f"\nSmallest normal float64: {np.finfo(np.float64).tiny}")
print(f"Largest float64:         {np.finfo(np.float64).max}")

print(f"\nfloat32 machine epsilon: {np.finfo(np.float32).eps}")
print(f"  = 2^(-23) = {2**-23}")

In [ ]:
# Dissect a float64 into sign, exponent, mantissa
def dissect_float64(x):
    """Show the IEEE 754 parts of a float64."""
    b = struct.pack('>d', x)
    bits = ''.join(f'{byte:08b}' for byte in b)
    sign = int(bits[0])
    exp_bits = bits[1:12]
    exp_val = int(exp_bits, 2) - 1023
    frac_bits = bits[12:]
    print(f"x = {x}")
    print(f"  sign: {sign} ({'−' if sign else '+'})")
    print(f"  exponent: {exp_bits} (= {int(exp_bits,2)} − 1023 = {exp_val})")
    print(f"  fraction: {frac_bits[:20]}... ({52} bits)")
    print()

dissect_float64(1.0)
dissect_float64(1.0 + eps)
dissect_float64(-0.1)

### Relative vs. Absolute Spacing

In [ ]:
# The gap between a float and its next neighbor
test_values = [1e-10, 1e-5, 1.0, 1e5, 1e10, 1e15]
print(f"{'Value':>15s}  {'Gap (absolute)':>20s}  {'Gap (relative)':>20s}")
print('-' * 60)
for x in test_values:
    gap = np.spacing(x)
    print(f"{x:15.2e}  {gap:20.2e}  {gap/x:20.2e}")

print(f"\nRelative gap is always ~eps = {eps:.2e}")

### Catastrophic Cancellation: $\sqrt{1+x} - 1$

In [ ]:
x_vals = np.logspace(-1, -16, 50)

# Unstable formula
f_unstable = np.sqrt(1 + x_vals) - 1

# Stable formula (rationalized)
f_stable = x_vals / (np.sqrt(1 + x_vals) + 1)

# "Exact" using higher precision concept: for small x, f(x) ≈ x/2
# More precisely, use the stable formula as reference
rel_err_unstable = np.abs(f_unstable - f_stable) / np.abs(f_stable)

plt.figure(figsize=(8, 5))
plt.loglog(x_vals, rel_err_unstable, 'r.-', markersize=4,
           label=r'$\sqrt{1+x} - 1$ (unstable)')
plt.axhline(eps, color='gray', linestyle='--', alpha=0.5, label=r'$\epsilon_{\mathrm{machine}}$')
plt.xlabel('$x$')
plt.ylabel('Relative error vs stable formula')
plt.title(r'Catastrophic cancellation in $\sqrt{1+x} - 1$')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"At x = 1e-14:")
x = 1e-14
print(f"  Unstable: {np.sqrt(1+x) - 1:.16e}")
print(f"  Stable:   {x / (np.sqrt(1+x) + 1):.16e}")

### Quadratic Formula Cancellation

In [ ]:
# x^2 - 10^8 x + 1 = 0
a, b_coeff, c = 1.0, -1e8, 1.0
disc = b_coeff**2 - 4*a*c

# Standard formula (cancellation in r2)
r1_std = (-b_coeff + np.sqrt(disc)) / (2*a)
r2_std = (-b_coeff - np.sqrt(disc)) / (2*a)

# Stable: compute r1 via non-cancelling branch, r2 via Vieta's
r1_stable = (-b_coeff + np.sqrt(disc)) / (2*a)
r2_stable = c / (a * r1_stable)  # Vieta: r1 * r2 = c/a

r2_exact = 1e-8  # approximately

print(f"x^2 - 10^8 x + 1 = 0")
print(f"\nStandard formula:")
print(f"  r1 = {r1_std:.10e}")
print(f"  r2 = {r2_std:.10e}  (should be ~1e-8)")
print(f"\nStable (Vieta's):")
print(f"  r1 = {r1_stable:.10e}")
print(f"  r2 = {r2_stable:.10e}  (much better!)")
print(f"\nRelative error in r2:")
print(f"  Standard: {abs(r2_std - r2_exact)/r2_exact:.2e}")
print(f"  Stable:   {abs(r2_stable - r2_exact)/r2_exact:.2e}")

### Floating Point Is Not Associative

In [ ]:
a = 1.0
b = 1e-16
c = 1e-16

result1 = (a + b) + c
result2 = a + (b + c)

print(f"a = {a}, b = {b}, c = {c}")
print(f"(a + b) + c = {result1:.20f}")
print(f"a + (b + c) = {result2:.20f}")
print(f"\nDifference:  {result2 - result1:.2e}")
print(f"b + c exact: {b + c:.2e}")
print(f"\nThe second ordering is more accurate.")
print(f"Adding small numbers first preserves more information.")

### Connection: $\kappa \cdot \epsilon_{\mathrm{machine}}$ = Achievable Accuracy

In [ ]:
np.random.seed(0)
ns = [5, 10, 15, 20, 30, 50]

print(f"{'n':>4s}  {'kappa(H_n)':>14s}  {'Predicted digits':>18s}  {'Actual rel error':>18s}  {'Actual digits':>14s}")
print('-' * 75)

for n in ns:
    H = hilbert(n)
    kap = np.linalg.cond(H)
    x_true = np.ones(n)
    b = H @ x_true
    x_comp = np.linalg.solve(H, b)
    rel_err = np.linalg.norm(x_comp - x_true) / np.linalg.norm(x_true)
    predicted = max(0, 16 - np.log10(kap))
    actual = max(0, -np.log10(rel_err + 1e-20))
    print(f"{n:4d}  {kap:14.2e}  {predicted:18.1f}  {rel_err:18.2e}  {actual:14.1f}")

### Visualizing the Floating Point Number Line

In [ ]:
# Show float distribution near powers of 2
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=False)

for ax, center, label in zip(axes,
    [1.0, 8.0, 1024.0],
    ['Near 1', 'Near 8', 'Near 1024']):

    # Generate consecutive floats
    floats = [center]
    x = center
    for _ in range(30):
        x = np.nextafter(x, np.inf)
        floats.append(x)
    floats = np.array(floats)
    gaps = np.diff(floats)

    ax.plot(floats, np.zeros_like(floats), '|', markersize=15, color='blue')
    ax.set_title(f'{label}: gap = {gaps[0]:.2e}', fontsize=10)
    ax.set_yticks([])
    ax.ticklabel_format(useOffset=False)

plt.suptitle('Floating point numbers are NOT evenly spaced', fontsize=12)
plt.tight_layout()
plt.show()